In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, RocCurveDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

## Data Loading

In [ ]:
X_train = pd.read_csv("../data/X_train.csv")
Y_train = pd.read_csv("../data/Y_train.csv").values.ravel()
X_test = pd.read_csv("../data/X_test.csv")
Y_test = pd.read_csv("../data/Y_test.csv").values.ravel()

In [ ]:
print(X_train.shape, Y_train.shape, X_test.shape, Y_test.shape)

## Model Training

In [ ]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, Y_train)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, Y_train)

xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, Y_train)

## Model Prediction

In [ ]:
lr_predict = lr_model.predict(X_test)
rf_predict = rf_model.predict(X_test)
xgb_predict = xgb_model.predict(X_test)

### Classification Report

In [ ]:
lr_cr = classification_report(Y_test, lr_predict)
print("Classification Report for Logistic Regression Model", lr_cr)

In [ ]:
rf_cr = classification_report(Y_test, rf_predict)
print("Classification Report for Random Forest Model", rf_cr)

In [ ]:
xgb_cr = classification_report(Y_test, xgb_predict)
print("Classification Report for XGBoost Model", xgb_cr)

### Confusion Matrix

In [ ]:
lr_cm = confusion_matrix(Y_test, lr_predict)
rf_cm = confusion_matrix(Y_test, rf_predict)
xgb_cm = confusion_matrix(Y_test, xgb_predict)
print("Confusion Matrix for Logistic Regression Model", lr_cm)
print("Confusion Matrix for Random Forest Model", rf_cm)
print("Confusion Matrix for XGBoost Model", xgb_cm)

### ROC-AUC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(6,5))
RocCurveDisplay.from_estimator(lr_model, X_test, Y_test, ax=ax)

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance level (AUC = 0.5)")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6,5))
RocCurveDisplay.from_estimator(rf_model, X_test, Y_test, ax=ax)

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance level (AUC = 0.5)")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6,5))
RocCurveDisplay.from_estimator(xgb_model, X_test, Y_test, ax=ax)

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance level (AUC = 0.5)")
ax.legend()
plt.show()



| Model | AUC Score |
| --- | --- |
| Logistic Regression | 0.83 |
| Random Forest | 0.82 |
| XGBoost | 0.83 |

- All 3 models perform significantly better than random guessing (dashed line at AUC = 0.5), with Logistic Regression and XGBoost tied at AUC = 0.83 and Random Forest slightly behind at 0.82.
- The curves rise steeply toward the top-left corner early on (meaning all 3 models are good at catching churned customers without too many false alarms)
- XGBoost is selected as the final model due to its highest overall prediction count and equal AUC score, making it the most reliable choice for production deployment.

## Finetuning XGBoost

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2]
}

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    verbose=1
)

grid_search.fit(X_train, Y_train)

# Print best parameters and score
print("Best Parameters:", grid_search.best_params_)
print("Best AUC Score:", grid_search.best_score_)

- After testing 27 combinations of hyperparameters across 5 cross-validation folds (135 fits total), GridSearchCV identified the optimal XGBoost configuration as learning rate of 0.1, max depth of 3, and 100 estimators.
- This tuning improved the AUC score from 0.83 to 0.852
- A meaningful gain that means the model is now better at distinguishing between customers who will churn and those who won't, making it more reliable for production use.

## Training XGBoost with BEST parameters

In [ ]:
best_xgb = XGBClassifier(
    learning_rate=grid_search.best_params_['learning_rate'],
    max_depth=grid_search.best_params_['max_depth'],
    n_estimators=grid_search.best_params_['n_estimators'],
    random_state=42
)
best_xgb.fit(X_train, Y_train)

In [ ]:
y_pred_best = best_xgb.predict(X_test)
print("Classification Report for Tuned XGBoost:")
print(classification_report(Y_test, y_pred_best))

- The tuned XGBoost model achieved an overall accuracy of 79% with an AUC of 0.852.
- It performs strongly at identifying customers who will stay (90% recall) but struggles to catch all churners (48% recall), which is expected given the dataset imbalance of 73.4% non-churners vs 26.6% churners.
- This class imbalance will be flagged as a known limitation in the project documentation, with SMOTE oversampling identified as a potential improvement in future iterations.

## Saving BEST Model

In [ ]:
model_path = os.path.join(os.getcwd(), '..', 'data', 'best_model.pkl')
joblib.dump(best_xgb, model_path)
print(f"Model saved to: {model_path}")